# Tarefa (executada pelo SLURM via papermill; não abra à mão)

Treina **um** modelo: uma configuração, uma dobra externa, uma dobra interna. Os parâmetros abaixo são
preenchidos pelo `job_tarefas.sh`. A cópia executada fica em `<lote>/notebooks/tarefa_<n>.ipynb`.
Se a tarefa for interrompida, o SLURM a recoloca na fila e o treino continua do `last.pt`.

In [ ]:
LOTE = ""        # pasta do lote
TAREFA = 0       # índice da tarefa (SLURM_ARRAY_TASK_ID)
DADOS = ""       # conjunto descompactado no disco local do nó
WANDB_MODE = None  # None = online (com recuo automático para offline); "disabled" = sem W&B

In [ ]:
import json
from pathlib import Path
from macdeg.ambiente import PROJECT, load_dotenv, which_secrets
from macdeg.cluster import run_task, task_dir

load_dotenv()   # $MACDEG_ENV: o lançador aponta para o .env do clone no $HOME (só o caminho)
lote = json.loads((Path(LOTE) / "lote.json").read_text())
tarefa = lote["tarefas"][TAREFA]
print("tarefa:", tarefa, "| segredos:", which_secrets())

In [ ]:
metricas = run_task(Path(LOTE), TAREFA, DADOS or None, WANDB_MODE)
saida = task_dir(Path(LOTE), tarefa)
m = json.loads((saida / "metrics.json").read_text())
print(f"AUROC de validação {m['auroc']:.4f} {m['auroc_ic95']} | régua da origem {m['regua_origem_auroc']:.4f} "
      f"| melhor época {m['best_epoch']} de {m['epochs_run']} | GPU {m['gpu']} | lote {m['micro_batch']}")
print("por base:", {b: round(v["auroc"], 4) if v["auroc"] is not None else None for b, v in m["por_base"].items()})